# Book Club Decision Log — memweave Temporal Decay Demo

Shows how temporal decay keeps agent memory fresh by penalising old files at query time — no rewriting of `.md` files needed.

## What this notebook demonstrates

| Concept | How it appears here |
|---|---|
| **Evergreen file** | `memory/club_info.md` — always full score, immune to decay |
| **Dated files** | `memory/YYYY-MM-DD.md` — score decays exponentially with age |
| **Temporal decay** | Old genre decisions rank lower; today's notes float to the top |
| **Agent comparison** | Agent A (no decay) vs Agent B (with decay) — same question, different context |

## Decay formula

```
multiplier = exp(−ln(2) / half_life_days × age_days)
```

At `age = 0` the multiplier is `1.0`. At `age = half_life_days` it is `0.5`. Evergreen files always get `1.0`.

## 1. Installation

Install `memweave` and set your API key as an environment variable — memweave works with any LiteLLM-compatible provider:

```bash
# OpenAI
export OPENAI_API_KEY="sk-..."

# Gemini
export GEMINI_API_KEY="..."

# Anthropic
export ANTHROPIC_API_KEY="sk-ant-..."
```

In [ ]:
%pip install memweave

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""  # set your key here

## 2. Imports & Configuration

Swap `EMBEDDING_MODEL` and `LLM_MODEL` for any [LiteLLM-supported model](https://docs.litellm.ai/docs/providers)
(e.g. `"gemini/gemini-2.5-flash"`, `"anthropic/claude-haiku-4-5"`).

In [ ]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path

import litellm

from memweave import EmbeddingConfig, MemoryConfig, MemWeave

EMBEDDING_MODEL = "text-embedding-3-small"  # swap for any LiteLLM-compatible embedding model
LLM_MODEL       = "gpt-4o-mini"             # swap for any LiteLLM-compatible model

WORKSPACE      = Path.cwd() / "workspace"
TODAY          = date.today()
HALF_LIFE_DAYS = 90  # score halves every 90 days

# Query that produces a clear Agent A vs Agent B difference:
# Without decay, older "vote" files rank higher than today's notes.
# With decay, today's notes float to rank 1.
AGENT_QUERY = "What genre did the club vote on most recently?"

SYSTEM_PROMPT = (
    "You are a helpful book club assistant. "
    "Answer the question using ONLY the context provided below. "
    "Be concise — two sentences maximum. "
    "If the context contains conflicting information, prefer the most recent source."
)

## 3. Create Sample Dataset

This cell writes 9 files to `workspace/memory/` — 1 evergreen, 7 historical dated files, and today's notes.

| File | Age | Content |
|---|---|---|
| `club_info.md` | Evergreen | Standing info: members, schedule, voting rules |
| `2024-10-05.md` | ~18 months | Fantasy vote — *The Name of the Wind* |
| `2024-12-18.md` | ~16 months | Historical fiction vote — *The Pillars of the Earth* |
| `2025-02-22.md` | ~14 months | Venue change: Café → Maplewood Library |
| `2025-05-10.md` | ~11 months | Mystery vote — *Gone Girl* |
| `2025-08-14.md` | ~8 months | New members joined, pace change |
| `2025-11-03.md` | ~5 months | Non-fiction vote — *Educated* |
| `2025-12-30.md` | ~3 months | 2026 plan: Sci-fi — *Project Hail Mary* |
| `{today}.md` | Today | Currently reading *Project Hail Mary*; literary fiction next |

Files are only written if they don't already exist — re-running the cell is safe.

In [ ]:
def write_memory_files() -> None:
    mem_dir = WORKSPACE / "memory"
    mem_dir.mkdir(parents=True, exist_ok=True)

    today_heading = TODAY.strftime("%B %-d, %Y")
    next_meeting  = (TODAY + timedelta(weeks=6)).strftime("%B %-d, %Y")

    files = {
        "club_info.md": "\n".join([
            "# Book Club — Evergreen Info",
            "",
            "**Members:** Alice, Ben, Clara, David, Evelyn (5 active members)",
            "**Meeting schedule:** First Saturday of every month, 3 pm at Maplewood Public Library",
            "**Voting rule:** Majority vote (3 of 5) picks the next genre and book title",
            "**Reading pace:** One book every six weeks; members may nominate up to two titles per session",
            "**Contact:** bookclub@example.com",
            "",
        ]),
        "2024-10-05.md": "\n".join([
            "# Book Club Meeting — October 2024",
            "",
            "**When:** Thursday evening, 7 pm",
            "**Where:** Brew & Read Café, 12 Elm Street (back room)",
            "**Genre vote for Q4 2024:** Fantasy (4-1 majority)",
            "**Selected title:** *The Name of the Wind* by Patrick Rothfuss",
            "**Notes:** David proposed epic fantasy after the group felt the previous reads were too",
            "heavy on non-fiction. Clara was the sole dissenting vote, preferring historical fiction.",
            "We agreed to revisit historical fiction in Q1 2025.",
            "",
        ]),
        "2024-12-18.md": "\n".join([
            "# Book Club Meeting — December 2024",
            "",
            "**When:** Thursday evening, 7 pm",
            "**Where:** Brew & Read Café, 12 Elm Street (back room)",
            "**Year-end review:** Fantasy quarter was well received (avg rating 4.2 / 5).",
            "**Genre vote for Q1 2025:** Historical fiction (unanimous — Clara's long-standing request).",
            "**Selected title:** *The Pillars of the Earth* by Ken Follett",
            "**Notes:** Evelyn suggested we invite a guest speaker for the historical fiction month.",
            "Alice volunteered to find a local historian.",
            "",
        ]),
        "2025-02-22.md": "\n".join([
            "# Book Club Meeting — February 2025",
            "",
            "**Logistics change:** Meeting moved from Thursday evenings at Brew & Read Café",
            "to Saturday afternoons at Maplewood Public Library (meeting room B).",
            "Reason: three members have conflicting Thursday commitments; café no longer available.",
            "**New schedule from March onwards:** First Saturday of the month, 3 pm, library.",
            "**Reading update:** Halfway through *The Pillars of the Earth* — strong engagement.",
            "**Upcoming genre vote:** Mystery / thriller proposed for spring (Ben's suggestion).",
            "",
        ]),
        "2025-05-10.md": "\n".join([
            "# Book Club Meeting — May 2025",
            "",
            "**When:** Saturday afternoon, 3 pm",
            "**Where:** Maplewood Public Library, meeting room B",
            "**Genre vote for summer 2025:** Mystery / thriller (3-2 majority).",
            "**Selected title:** *Gone Girl* by Gillian Flynn",
            "**Notes:** Evelyn pushed for literary fiction but was outvoted. David suggested we",
            "alternate between lighter reads and literary fiction going forward.",
            "Clara is stepping back to guest status for June due to travel commitments.",
            "",
        ]),
        "2025-08-14.md": "\n".join([
            "# Book Club Meeting — August 2025",
            "",
            "**Three new associate members joined:** Fiona, George, Hana (trial period: 3 months).",
            "**Reading pace update:** Reduced from one book per month to one every six weeks",
            "to accommodate busier schedules over autumn.",
            "**Current read:** Still mystery / thriller; finishing *Big Little Lies* by Liane Moriarty.",
            "**Upcoming vote:** Genre for Q4 2025 to be decided at October meeting.",
            "",
        ]),
        "2025-11-03.md": "\n".join([
            "# Book Club Meeting — November 2025",
            "",
            "**Genre vote for Q4 2025:** Non-fiction (4-1 majority).",
            "**Selected title:** *Educated* by Tara Westover",
            "**Notes:** The group felt it had been two years since the last non-fiction pick.",
            "Fiona, George, and Hana are now full members after completing their trial period.",
            "Meeting cadence remains every-six-weeks until the new year.",
            "",
        ]),
        "2025-12-30.md": "\n".join([
            "# Book Club End-of-Year Review — December 2025",
            "",
            "**2025 summary:** Fantasy, historical fiction, mystery, and non-fiction covered.",
            "**2026 reading plan:** The group voted to start the year with science fiction (5-0).",
            "**Selected title for January 2026:** *Project Hail Mary* by Andy Weir",
            "**Hosting rota for 2026:** Alice (Jan), Ben (Mar), Clara (May), David (Jul),",
            "Evelyn (Sep), Fiona (Nov).",
            "**New standing rule:** The host for each session picks a short-story bonus read",
            "(under 50 pages) to warm up the discussion.",
            "",
        ]),
        f"{TODAY.isoformat()}.md": "\n".join([
            f"# Book Club Meeting — {today_heading} (Today)",
            "",
            "**Current read:** *Project Hail Mary* by Andy Weir (science fiction) — on track.",
            "**Next genre vote:** The group is leaning toward literary fiction for the next pick.",
            "Evelyn has been campaigning for literary fiction for over a year; likely to pass.",
            "**Proposed titles:** *Normal People* by Sally Rooney, *Pachinko* by Min Jin Lee.",
            f"**Next meeting:** {next_meeting} hosted by Ben.",
            "**Action items:**",
            "- All members to read final 100 pages before next meeting.",
            "- Ben to confirm library room booking by end of this week.",
            "",
        ]),
    }

    written = []
    for filename, content in files.items():
        path = mem_dir / filename
        if not path.exists():
            path.write_text(content, encoding="utf-8")
            written.append(filename)

    print(f"Memory files ({len(files)} total, {len(written)} newly written):")
    for name in sorted(files):
        label = " ← TODAY" if name.startswith(TODAY.isoformat()) else (
            " ← EVERGREEN" if name == "club_info.md" else ""
        )
        status = "" if name in written else "  [already exists]"
        print(f"  memory/{name}{label}{status}")


write_memory_files()

## 4. Index Memory

`mem.index()` scans `workspace/memory/`, embeds new or changed files, and caches them in a local SQLite database.
Re-running is instant — only changed files are re-embedded (content is SHA-256 hashed).

In [ ]:
config = MemoryConfig(
    workspace_dir=WORKSPACE,
    embedding=EmbeddingConfig(model=EMBEDDING_MODEL),
)

async with MemWeave(config) as mem:
    result = await mem.index()
    print(f"Indexed {result.files_indexed} files ({result.files_skipped} skipped).")

## 5. Two Agents, One Question

The same question is answered by two agents with different retrieval strategies:

| Agent | Decay | What happens |
|---|---|---|
| **Agent A** | None | Top files by raw semantic similarity — older "vote" files score higher than today's notes |
| **Agent B** | `half_life=90d` | Age penalty crushes old files — today's notes float to rank 1 |

The retrieved context for each agent is printed alongside the LLM's answer.
The answers differ because each agent sees a different slice of the same memory.

In [ ]:
async def ask_agent(mem: MemWeave, query: str, *, decay: bool) -> tuple[list, str]:
    """Search memory and generate an LLM answer grounded in the retrieved context."""
    kwargs: dict = {"max_results": 3, "min_score": 0.1}
    if decay:
        kwargs["decay_half_life_days"] = float(HALF_LIFE_DAYS)

    results = await mem.search(query, **kwargs)

    context = "\n\n---\n\n".join(
        f"Source: {Path(r.path).name}\n\n{r.snippet}" for r in results
    )
    messages = [
        {"role": "system", "content": f"{SYSTEM_PROMPT}\n\nContext:\n\n{context}"},
        {"role": "user",   "content": query},
    ]
    response = await litellm.acompletion(model=LLM_MODEL, messages=messages, max_tokens=120)
    return results, response.choices[0].message.content.strip()


def print_agent_result(label: str, results: list, answer: str) -> None:
    print(label)
    print("  Context retrieved:")
    for r in results:
        print(f"    [{r.score:.3f}]  {Path(r.path).name}")
    print(f"  Answer:\n    {answer}\n")


print(f'Question: "{AGENT_QUERY}"\n')

async with MemWeave(config) as mem:
    results_a, answer_a = await ask_agent(mem, AGENT_QUERY, decay=False)
    results_b, answer_b = await ask_agent(mem, AGENT_QUERY, decay=True)

print_agent_result("Agent A — no temporal decay (raw relevance):", results_a, answer_a)
print_agent_result(f"Agent B — with temporal decay (half_life={HALF_LIFE_DAYS}d):", results_b, answer_b)

## Key Observations

- **Agent A (no decay)**: today's file doesn't appear in the top 3 — raw semantic similarity
  ranks older "vote" files higher. It answers **"Non-fiction"** (the Nov 2025 vote), which
  is factually stale.

- **Agent B (with decay)**: today's file floats to rank 1 after the age penalty crushes the
  old files. It correctly answers **"Science fiction"** (the most recent actual vote from the
  Dec 2025 year-end review).

- **`club_info.md` (evergreen)** surfaces in Agent B's context at full score — immunity to
  decay is determined by the file path (any non-dated file under `memory/`), not the content.

- **Old 2024 files disappear from Agent B's context entirely** — after the 90-day half-life
  they are outranked by recent and evergreen files.

- **Scale effect**: with months or years of accumulated history, Agent A's context fills with
  increasingly stale votes and decisions. Decay's advantage grows the larger the memory gets.

## Try It Yourself

Swap `AGENT_QUERY` in the config cell above and re-run the agent cell to explore different behaviours.

| Query | What to expect |
|---|---|
| `"Who are the current members of the book club?"` | `club_info.md` (evergreen) ranks #1 in **both** agents — demonstrates that evergreen immunity is robust regardless of decay setting |
| `"When and where does the book club meet?"` | Agent A: old Café files compete with `club_info.md`. Agent B: `club_info.md` and today's file dominate — stale venue info disappears |
| `"What action items are outstanding?"` | Agent A surfaces action items from every meeting equally, including long-completed 2024 tasks. Agent B surfaces only recent ones |
| `"What book should we read next?"` | Agent A retrieves every past title suggestion equally. Agent B surfaces today's proposed titles (*Normal People*, *Pachinko*) above older ones |

You can also tune `HALF_LIFE_DAYS` to see how the half-life affects the results:
- **`HALF_LIFE_DAYS = 30`** — aggressive decay; files older than a month drop sharply
- **`HALF_LIFE_DAYS = 180`** — gentle decay; older files retain more of their score